## Using Python to Perform Extract-Transform-Load (ETL Processing)

#### Import the Necessary Libraries

In [ ]:
import os
import json
import numpy
import datetime
import certifi
import pandas as pd

import pymongo
import sqlalchemy
from sqlalchemy import create_engine, text

In [ ]:
print(f"Running SQL Alchemy Version: {sqlalchemy.__version__}")
print(f"Running PyMongo Version: {pymongo.__version__}")

#### Declare & Assign Connection Variables for the MySQL Server & Databases with which You'll be Working 

In [ ]:
host_name = "localhost"
port = "3306"
user_id = "root"
pwd = "NewPasswordHere"

src_dbname = "classicmodels"
dst_dbname = "classicmodels_dw2"

#### Declare & Assign Connection Variables for the MongoDB Server, the MySQL Server & Databases with which You'll be Working 

In [ ]:
mysql_args = {
    "uid" : "root",
    "pwd" : "NewPasswordHere",
    "hostname" : "localhost",  #"wna8fw-mysql.mysql.database.azure.com",
    "dbname" : "classicmodels_dw2"
}

# The 'cluster_location' must either be "atlas" or "local".
mongodb_args = {
    "user_name" : "",
    "password" : "",
    "cluster_name" : "",
    "cluster_subnet" : "",
    "cluster_location" : "local", # "local"
    "db_name" : "classicmodels_mongo"
}


#### Define Functions for Getting Data From and Setting Data Into Databases

In [ ]:
def get_dataframe(user_id, pwd, host_name, db_name, sql_query):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    dframe = pd.read_sql(sql_query, connection);
    connection.close()
    
    return dframe


def set_dataframe(user_id, pwd, host_name, db_name, df, table_name, pk_column, db_operation):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
    
    '''Invoke the Pandas DataFrame .to_sql( ) function to either create, or append to, a table'''
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
                    
        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')

    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")
    
    db_connection.close()


def get_mongo_client(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
    
    else:
        if args["cluster_location"] == "atlas":
            connect_str = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
            
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
        
    return client


def get_mongo_dataframe(mongo_client, db_name, collection, query):
    '''Query MongoDB, and fill a python list with documents to create a DataFrame'''
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()
    
    return dframe


def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()

#### Create the New Data Warehouse database, and to Use it, Switch the Connection Context.


In [ ]:
conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}"
print(conn_str)

sqlEngine = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()

connection.execute(text(f"DROP DATABASE IF EXISTS `{dst_dbname}`;"))
connection.execute(text(f"CREATE DATABASE `{dst_dbname}`;"))
connection.execute(text(f"USE {dst_dbname};"))

connection.close()

#### Populate MongoDB with Source Data

In [ ]:
client = get_mongo_client(**mongodb_args)

# Gets the path of the Current Working Directory for this Notebook,
# and then Appends the 'data' directory.
data_dir = os.path.join(os.getcwd(), 'data')

json_files = {
              "products" : 'classicmodels_products.json'
             }

set_mongo_collections(client, mongodb_args["db_name"], data_dir, json_files)         

### Create & Populate the Dimension Tables

In [ ]:
# Extract data from the "products" collection
client = get_mongo_client(**mongodb_args)

query = {}  # Select all documents
collection = "products"

df_products = get_mongo_dataframe(client, mongodb_args["db_name"], collection, query)

df_products.head(2)


In [ ]:
# Customers
sql_customers = "SELECT * FROM classicmodels.customers;"
df_customers = get_dataframe(user_id, pwd, host_name, src_dbname, sql_customers)
df_customers.head(2)

In [ ]:
# Employees
sql_employees = "SELECT * FROM classicmodels.employees;"
df_employees = get_dataframe(user_id, pwd, host_name, src_dbname, sql_employees)
df_employees.head(2)

#### 1.2. Create the Date Dimension Table
**execute the script "Project1_dim_date.sql"** that creates and populates a **Date Dimension** table. 

#### 1.3. Perform Any Necessary Transformations

In [ ]:
# Customers
# 1. Create a List that enumerates the names of each column you wish to remove (drop) from the Pandas DataFrame
drop_cols = ['contactLastName', 'contactFirstName']
df_customers.drop(drop_cols, axis=1, inplace=True)

# 2. Rename the "id" column to reflect the entity as it will serve as the business key for lookup operations
df_customers.rename(columns={"customerNumber":"customer_id"}, inplace=True)

# 3. Display the first 2 rows of the dataframe to validate your work
df_customers.head(2)

In [ ]:
# Employees
# 1. Create a List that enumerates the names of each column you wish to remove (drop) from the Pandas DataFrame
drop_cols = ['extension', 'officeCode']
df_employees.drop(drop_cols, axis=1, inplace=True)

# 2. Rename the "id" column to reflect the entity as it will serve as the business key for lookup operations
df_employees.rename(columns={"employeeNumber":"employee_id"}, inplace=True)

# 3. Display the first 2 rows of the dataframe to validate your work
df_employees.head(2)

In [ ]:
# Products
# 1. Create a List that enumerates the names of each column you wish to remove (drop) from the Pandas DataFrame
drop_cols = ['productDescription', 'productScale']
df_products.drop(drop_cols, axis=1, inplace=True)

# 2. Rename the "id" column to reflect the entity as it will serve as the business key for lookup operations
df_products.rename(columns={"productCode":"product_id"}, inplace=True)

# 3. Display the first 2 rows of the dataframe to validate your work
df_products.head(2)

#### 1.4. Load the Transformed DataFrames into the New Data Warehouse by Creating New Tables

In [ ]:
db_operation = "insert"

tables = [('dim_customers', df_customers, 'customer_key'),
          ('dim_employees', df_employees, 'employee_key'),
          ('dim_products', df_products, 'product_key')]

In [ ]:
for table_name, dataframe, primary_key in tables:
    set_dataframe(user_id, pwd, host_name, dst_dbname, dataframe, table_name, primary_key, db_operation)

### 2.0. Create & Populate the Fact Table

In [ ]:
sql_fact_order_lines = """
    SELECT
      o.orderNumber,
      od.orderLineNumber,
      o.customerNumber,
      od.productCode,
      c.salesRepEmployeeNumber,
      o.orderDate,
      od.quantityOrdered,
      od.priceEach,
      (od.quantityOrdered * od.priceEach) AS extendedAmount
    FROM classicmodels.orders AS o
    JOIN classicmodels.orderdetails AS od
      ON o.orderNumber = od.orderNumber
    JOIN classicmodels.customers AS c
      ON o.customerNumber = c.customerNumber;
"""

df_fact_order_lines = get_dataframe(user_id, pwd, host_name, src_dbname, sql_fact_order_lines)
df_fact_order_lines.head(2)

### Instead, implement the solution using Pandas DataFrames to craft the table

#### 2.1. Get all the data from each of the tables involved

In [ ]:
sql_orders = "SELECT * FROM classicmodels.orders;"
df_orders = get_dataframe(user_id, pwd, host_name, src_dbname, sql_orders)
df_orders.rename(columns={"orderNumber": "order_id"}, inplace=True)
df_orders.head(2)

In [ ]:
# 1. SELECT all columns from the classicmodels.orderdetails table to create the "df_order_details" dataframe
sql_orderdetails = "SELECT * FROM classicmodels.orderdetails;"
df_orderdetails = get_dataframe(user_id, pwd, host_name, src_dbname, sql_orderdetails)
# 2. Rename the "orderNumber" column to "order_detail_id"
df_orderdetails.rename(columns={"orderNumber": "order_id"}, inplace=True)
df_orderdetails.rename(columns={"orderLineNumber": "order_line_id"}, inplace=True)
# 3. Display the first two rows of the DataFrame to validate your work
df_orderdetails.head(2)

#### 2.2. Join the Orders and OrderDetails DataFrames

In [ ]:
print(df_orderdetails.columns.tolist())

df_ordersdetails = pd.merge(
    df_orders,
    df_orderdetails,
    on='order_id',
    how='inner'
)


df_ordersdetails.head(2)


In [ ]:
df_fact_order_lines.shape

#### 2.3. Lookup the Primary Keys from the Dimension Tables


##### 2.3.1. Fetch the Primary Key and Business Key from the Date Dimension Table.


In [ ]:
sql_dim_date = "SELECT date_key, full_date FROM classicmodels_dw2.dim_date;"
df_dim_date = get_dataframe(user_id, pwd, host_name, src_dbname, sql_dim_date)
df_dim_date.full_date = df_dim_date.full_date.astype('datetime64[ns]').dt.date
df_dim_date.head(2)

##### 2.3.2. Next, lookup the Surrogate Primary Key values using the corresponding Business Key,

In [ ]:
# Lookup the Surrogate Primary Key (date_key) that Corresponds to the "order_date" Column.
df_fact_order_lines.rename(columns={"orderDate": "order_date"}, inplace=True)

df_dim_order_date = df_dim_date.rename(columns={"date_key" : "order_date_key", "full_date" : "order_date"})
df_fact_order_lines.order_date = df_fact_order_lines.order_date.astype('datetime64[ns]').dt.date

df_fact_order_lines = pd.merge(df_fact_order_lines, df_dim_order_date, on='order_date', how='left')
df_fact_order_lines.drop(['order_date'], axis=1, inplace=True)
df_fact_order_lines.head(2)

##### 2.3.3. First, fetch the Surrogate Primary Key and the Business Key from each of the remaining Dimension tables using the **get_dataframe()** function.

In [ ]:
# Select 'customer_key' and 'customer_id' from classicmodels_dw2.dim_customers
sql_dim_customers = "SELECT customer_key, customer_id FROM classicmodels_dw2.dim_customers;"
df_dim_customers = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_dim_customers)
df_dim_customers.head(2)

In [ ]:
# Employees
sql_dim_employees = "SELECT employee_key, employee_id FROM classicmodels_dw2.dim_employees;"
df_dim_employees = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_dim_employees)
df_dim_employees.head(2)

In [ ]:
# Products
sql_dim_products = "SELECT product_key, product_id FROM classicmodels_dw2.dim_products;"
df_dim_products = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_dim_products)
df_dim_products.head(2)

##### 2.3.4. Next, using the Business Keys, lookup the corresponding Surrogate Primary Key values in the Dimension tables

In [ ]:
# 1. Modify 'df_fact_orders' by merging it with 'df_dim_customers' on the 'customer_id' column
df_fact_order_lines.rename(columns={"customerNumber": "customer_id"}, inplace=True)
df_fact_order_lines = pd.merge( df_fact_order_lines, df_dim_customers, on='customer_id', how='left' )
# 2. Drop the 'customer_id' column
df_fact_order_lines.drop(['customer_id'], axis=1, inplace=True)
# 3. Display the first 2 rows of the dataframe to validate your work
df_fact_order_lines.head(2)

In [ ]:
# Repeat for the Employees dimension
df_fact_order_lines.rename(columns={"salesRepEmployeeNumber": "employee_id"}, inplace=True)
df_fact_order_lines = pd.merge( df_fact_order_lines, df_dim_employees, on='employee_id', how='left' ) 
df_fact_order_lines.drop(['employee_id'], axis=1, inplace=True) 
df_fact_order_lines.head(2)

In [ ]:
# Repeat for the Product dimension
df_fact_order_lines.rename(columns={"productCode": "product_id"}, inplace=True)
df_fact_order_lines = pd.merge( df_fact_order_lines, df_dim_products, on='product_id', how='left' )
df_fact_order_lines.drop(['product_id'], axis=1, inplace=True)
df_fact_order_lines.head(2)

#### 2.4. Write the DataFrame Back to the Database

In [ ]:
table_name = "fact_order_lines"
primary_key = "fact_order_key"
db_operation = "insert"

set_dataframe(user_id, pwd, host_name, dst_dbname, df_fact_order_lines, table_name, primary_key, db_operation)

### 3.0. Demonstrate that the New Data Warehouse Exists and Contains the Correct Data
To demonstrate the viability of your solution, author a SQL SELECT statement that returns:
- Each Customer’s Name
- The total amount of the order quantity associated with each customer
- The total amount of the order unit price associated with each customer


In [ ]:
sql_test = """
SELECT 
    c.customerName AS customer_name,
    SUM(f.quantityOrdered) AS total_quantity,
    SUM(f.priceEach) AS total_unit_price
FROM {0}.fact_order_lines AS f
INNER JOIN {0}.dim_customers AS c
    ON f.customer_key = c.customer_id
GROUP BY 
    c.customerName
ORDER BY 
    total_unit_price DESC;
""".format(dst_dbname)

df_test = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_test)
df_test.head()


### 3.1 Highest revenue product by customer name

In [ ]:
sql_test = """
SELECT
    c.customerName,
    p.productName,
    SUM(f.quantityOrdered) AS total_quantity,
    SUM(f.priceEach * f.quantityOrdered) AS total_revenue
FROM {0}.fact_order_lines AS f
JOIN {0}.dim_customers AS c
    ON f.customer_key = c.customer_id
JOIN {0}.dim_products AS p
    ON f.product_key = p.product_key
GROUP BY
    c.customerName,
    p.productName
ORDER BY
    total_revenue DESC;

""".format(dst_dbname)

df_test = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_test)
df_test.head()


In [ ]:
os.listdir("data/orders_stream")

In [ ]:
df_fact_order_lines.head()
